In [1]:
print('Installing zstd...')
!sudo apt-get update && sudo apt-get install -y zstd

Installing zstd...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,389 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,924 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:13 https://ppa.launchpadco

In [4]:
import os
import time
import subprocess
import requests

# Kill any existing Ollama processes to ensure a clean start
print("Stopping any existing Ollama processes...")
os.system("killall ollama || true")
time.sleep(2) # Give it a moment to stop

# Ensure zstd is installed and recognized before Ollama installation
print("Ensuring zstd is installed...")
!sudo apt-get update -qq && sudo apt-get install -y zstd -qq

# 1. Install Ollama (this will re-run but typically won't re-download if already installed)
print("Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the server robustly in the background
print("Starting Ollama in the background...")
# Use subprocess.Popen for better control over the background process
# Redirecting stdout and stderr to os.devnull to keep the notebook output clean
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for Ollama server to start and become available
print("Waiting for Ollama server to start (up to 30 seconds)...")
start_time = time.time()
timeout = 30 # seconds
ollama_ready = False
api_url = "http://localhost:11434"

while time.time() - start_time < timeout:
    try:
        response = requests.get(api_url, timeout=1)
        if response.status_code == 200: # Ollama's health check usually returns 200
            ollama_ready = True
            print("Ollama server is ready!")
            break
    except requests.exceptions.ConnectionError:
        pass # Server not yet ready, keep trying
    except requests.exceptions.RequestException as e:
        print(f"Error checking Ollama status: {e}")
        # Continue trying even on other request errors
    time.sleep(1)

if not ollama_ready:
    # If the server didn't start, try to get logs from ollama_server.log or print error.
    # For simplicity, we'll just raise an error here.
    raise RuntimeError("Ollama server failed to start within the given timeout. Please check Ollama logs if available.")

# 3. Pull the model
print("Pulling the Llama 3 model...")
!ollama pull llama3
print("Ollama is ready for use!")

Stopping any existing Ollama processes...
Ensuring zstd is installed...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-

In [ ]:
import os
import json
import pandas as pd
import requests

class LLMRouterV1:
    def __init__(self, model_name='llama3', taxonomy_path='../../Data/taxonomy_v1.json'):
        self.model_name = model_name
        self.api_url = "http://localhost:11434/api/generate"

        # Load the taxonomy
        print(f"Loading taxonomy from {taxonomy_path}...")
        with open(taxonomy_path, 'r') as f:
            self.taxonomy = json.load(f)

    def build_system_prompt(self, domain):
        """Constructs the prompt using the specific domain labels from the taxonomy."""
        # The taxonomy has 'education' and 'healthcare' keys
        domain_key = domain.lower()
        if domain_key not in self.taxonomy['domains']:
            raise ValueError(f"Domain '{domain}' not found in taxonomy.")

        labels = self.taxonomy['domains'][domain_key]['labels']

        # Format labels into a readable string for the LLM
        labels_text = "\n".join([f"- {l['name']}: {l['definition']}" for l in labels])

        system_prompt = f"""You are an expert routing agent for a {domain_key} support system.
Your task is to classify the user's request into EXACTLY ONE of the following routing categories:

{labels_text}

Analyze the user's prompt carefully. You must output your response ONLY as a valid JSON object with the following exact keys:
{{
    "predicted_label": "The exact name of the label from the list above",
    "confidence_level": "High, Medium, or Low",
    "short_reason": "One short sentence explaining why you chose this label",
    "needs_clarification": true or false (use true if the prompt is too ambiguous, vague, or missing critical details)
}}
Do not include any markdown formatting, conversational text, or explanations outside of the JSON object.
"""
        return system_prompt

    def route_request(self, user_prompt, domain):
        """Sends the prompt to the local Ollama model and parses the JSON response."""
        system_prompt = self.build_system_prompt(domain)

        # Combine system instructions and the actual user request
        full_prompt = f"{system_prompt}\n\nUSER REQUEST:\n\"{user_prompt}\""

        payload = {
            "model": self.model_name,
            "prompt": full_prompt,
            "stream": False,
            "format": "json" # Forces Ollama to return valid JSON
        }

        try:
            response = requests.post(self.api_url, json=payload)
            response.raise_for_status()

            result_text = response.json().get("response", "{}")

            # Parse the string back into a Python dictionary
            parsed_result = json.loads(result_text)
            return parsed_result

        except Exception as e:
            print(f"Error calling LLM for prompt: '{user_prompt[:30]}...' -> {e}")
            # Fallback if the LLM fails or hallucinates bad JSON
            return {
                "predicted_label": "Error",
                "confidence_level": "Low",
                "short_reason": f"API Error: {str(e)}",
                "needs_clarification": True
            }

    def evaluate_benchmark(self, input_csv, output_csv):
        """Runs the LLM over the entire pilot benchmark and saves the results."""
        print(f"Loading benchmark data from {input_csv}...")
        df = pd.read_csv(input_csv)

        results = []

        print(f"Routing {len(df)} requests. This will take a few minutes depending on your hardware...")

        for index, row in df.iterrows():
            prompt_text = row['prompt']
            domain = row['domain']

            print(f"Processing [{index+1}/{len(df)}]: {prompt_text[:40]}...")

            # Ask the LLM to route it
            llm_output = self.route_request(prompt_text, domain)

            # Combine original row data with LLM predictions
            result_row = {
                "prompt_id": row.get('prompt_id', f"ID-{index}"),
                "domain": domain,
                "user_prompt": prompt_text,
                "gold_label": row.get('label', ''),
                "is_ambiguous_gold": row.get('is_ambiguous', ''),
                "predicted_label": llm_output.get("predicted_label", ""),
                "confidence_level": llm_output.get("confidence_level", ""),
                "needs_clarification_pred": llm_output.get("needs_clarification", False),
                "short_reason": llm_output.get("short_reason", "")
            }
            results.append(result_row)

        # Save to the new CSV
        results_df = pd.DataFrame(results)
        results_df.to_csv(output_csv, index=False)
        print(f"\nDone! Results saved to {output_csv}")

        # Calculate a quick accuracy metric
        correct = (results_df['gold_label'] == results_df['predicted_label']).sum()
        total = len(results_df)
        print(f"Rough Accuracy: {correct}/{total} ({(correct/total)*100:.1f}%)")

# ==========================================
# Run the evaluation
# ==========================================
if __name__ == "__main__":
    # Use known absolute paths in Colab environment
    taxonomy_path = "/content/taxonomy_v1.json"
    benchmark_path = "/content/v0_pilot_benchmark.csv"
    output_path = "/content/v1_llm_results.csv"

    print(f"Using taxonomy file at: {taxonomy_path}")

    # Initialize the router WITH the correct path
    router = LLMRouterV1(model_name='llama3', taxonomy_path=taxonomy_path)

    # Run the benchmark
    router.evaluate_benchmark(benchmark_path, output_path)

Using taxonomy file at: /content/taxonomy_v1.json
Loading taxonomy from /content/taxonomy_v1.json...
Loading benchmark data from /content/v0_pilot_benchmark.csv...
Routing 138 requests. This will take a few minutes depending on your hardware...
Processing [1/138]: "Can you explain the difference between ...
Processing [2/138]: "What exactly does 'Big O Notation' meas...
Processing [3/138]: "What is the difference between an Inter...
Processing [4/138]: "I don't understand how this works."...
Processing [5/138]: "Can you explain the thing about memory?...
Processing [6/138]: "Can you explain the difference between ...
Processing [7/138]: "I don't understand trees at all."...
Processing [8/138]: "Can you explain the exact step-by-step ...
Processing [9/138]: "I have been staring at the definition o...
